# PSZ - surface brightness profiles

Use `pyproffit` to extract and fit surface brightness profiles for PSZ. Make plots of fits.

TOC
- [Imports](##Imports)
- [Load data](##load-data)
    - [XMM-Newton](###XMM-Newton)
    - [Chandra](###Chandra)
- [Edge fits](#edge-fits)
    - [Define edges](#define-edges)
    - [XMM fits](#xmm-fits)
    - [Chandra fits](#chandra-fits)
- [Mach numbers](#mach-numbers)

## Imports

In [1]:
# Default library
from typing import get_args
import warnings

# Third party libraries
import numpy as np
import astropy.units as u
from astropy.coordinates import Angle
from astropy.wcs import FITSFixedWarning
import pyproffit


warnings.simplefilter("ignore", category=FITSFixedWarning)
warnings.filterwarnings("ignore", category=FutureWarning, module="arviz")


# Local libraries
from utils.utils_pyproffit import Mach_numbers, Edges, ModelType

## Load data

### XMM-Newton

In [2]:
xmm_dir = "./input_data/XMM_data/"
dat_xmm = pyproffit.Data(
    imglink=xmm_dir + "all_500_2000_count.img",  # image
    explink=xmm_dir + "all_effexpo_ms.img",  # exposure map
    bkglink=xmm_dir + "all_500_2000_bkg_sc.img",  # background map
)
# Load regions to be masked, including point sources
dat_xmm.region(xmm_dir + "exclusion_regions.reg")
# Mask areas that have an exposure lower than 10% of the maximum exposure
dat_xmm.exposure[dat_xmm.exposure < np.max(dat_xmm.exposure) * 0.1] = 0

Excluded 201 sources


### Chandra

In [3]:
chandra_dir = "./input_data/Chandra_data/"
dat_chandra = pyproffit.Data(
    imglink=chandra_dir + "acisi_0.5_2_b2_psrem.fits",
    explink=chandra_dir + "expmap_0123_b2_psrem.fits",
    bkglink=chandra_dir + "scbg_0123_0.5_2_psrem.fits",
)

In [4]:
dat_chandra.region(chandra_dir + "sources_mod.reg")
dat_chandra.exposure[dat_chandra.exposure < np.max(dat_chandra.exposure) * 0.1] = 0

Error: invalid format




---



## Edge fits

### Define edges

The regions below were defined via a combination of:
- Exploring the data with unsharp masked and GGM filtered images, created with a range of kernel sizes
- Exploring the data using `pyproffit`

Before defining the sectors in external region file, first explore the data directly with `pyproffit`. Discover (and confirm the presence of) significant edges by exploring different combinations of annulus properties, such as center coordinates, start and stop angle. 


In [5]:
# Define edges by name and region file
PSZ_edges = Edges(
    {
        "S edge": "./input_data/SB_S_sector.reg",
        "NW edge": "./input_data/SB_NW_sector.reg",
        "N edge": "./input_data/SB_N_sector.reg",
        "SW relic": "./input_data/SB_SW_relic_sector.reg",
        "NE relic": "./input_data/SB_NE_relic_sector.reg",
    }
)

In [6]:
# Define which models to fit
models: tuple[ModelType, ...] = get_args(ModelType)
print(models)

('bknpow', 'pow', 'beta')


After deciding on promising edges, explore their nature with `pyproffit`. 

For ``PSZ``, there are two classes of putative edges:
1. Central edges: located within 0.5 Mpc of the cluster core
    - Benefit from higher number statistics
    - Need ensure the drop is not explained by a simple drop in the cluster profile
    - Test whether a power-law, broken power-law as well as a beta model to confirm the nature of the putative edge
    - For the broken power-law model, given the good number statistics, the radial distance can be kept as a free parameter
2. Distant edges: located outside 0.5 Mpc of the cluster core, e.g. such as edges associated with radio relics; 
    - Number statistics is much poorer
    - Test whether a power-law or broken power-law provides the best fit
    - For the broken power-law model, given their large cluster centric location and the low number statistics, the radial distnace of the discontinuity is fixed to the location of the radio relic

### XMM fits

Fit models on *XMM-Newton* data. Given the higher number statistics in *XMM-Newton* data for ``PSZ``, we can leave the binning size to the default value of ``7''``. As discussed before, we fix the radial distance of the discontinuity only for the radio relics.

In [7]:
for name, edge in PSZ_edges.edges.items():
    for model in models:
        print(f"Now fitting {model} model to {name}")
        # fix the distance of the break only for the relics
        if "relic" in name and model == "bknpow":
            edge.fit_model(dat_xmm, model, "XMM", fix_rf=True)
        else:
            edge.fit_model(dat_xmm, model, "XMM")
        print()

Now fitting bknpow model to S edge
Corresponding pixels coordinates:  326.37473433929307 333.6060292823326
┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 19.36                      │              Nfcn = 298              │
│ EDM = 4.97e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴─────────────────────

/Users/astroe/Desktop/PSZ_GitHub/Photometric_analysis/utils/utils_pyproffit.py:414: RuntimeWarning: divide by zero encountered in divide
  chi = (profile.profile - tmod) / profile.eprof



Now fitting pow model to SW relic
Corresponding pixels coordinates:  323.60541720220374 336.16300698365313
┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 16.22                      │              Nfcn = 107              │
│ EDM = 1.03e-08 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴────────────────────

### Chandra fits

Fit models on *Chandra* data. Given the lower number statistics in the *Chandra* data for ``PSZ``, the binning was set for each edge size as to maximize S/N without trading off too much spatial resolution in the resulting binned profile. As discussed before, we fix the radial distance of the discontinuity only for the radio relics.

In [8]:
Chandra_bin_size: dict[str, Angle] = {
    "S edge": 10 * u.arcsec,
    "NW edge": 20 * u.arcsec,
    "N edge": 15 * u.arcsec,
    "SW relic": 30 * u.arcsec,
    "NE relic": 30 * u.arcsec,
}
for (name, edge), bs in zip(PSZ_edges.edges.items(), Chandra_bin_size.values()):
    for model in models:
        # fix the distance of the break only for the relics
        if "relic" in name and model == "bknpow":
            edge.fit_model(dat_chandra, model, "Chandra", bs=bs, fix_rf=True)
        else:
            edge.fit_model(dat_chandra, model, "Chandra", bs=bs)

Corresponding pixels coordinates:  895.5803433794657 791.8479145041406
┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 10.54                      │              Nfcn = 783              │
│ EDM = 4.94e-06 (Goal: 0.0002)    │            time = 0.1 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────┬───

/opt/miniconda3/envs/PSZ_env/lib/python3.10/site-packages/pyproffit/models.py:87: RuntimeWarning: invalid value encountered in power
  out = n2 * np.power(x / pivot, -alpha) + c2


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 25.65                      │              Nfcn = 692              │
│ EDM = 6.44e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬─────

/Users/astroe/Desktop/PSZ_GitHub/Photometric_analysis/utils/utils_pyproffit.py:456: UserWarning: Attempt to set non-positive ylim on a log-scaled axis will be ignored.
  ax1.set_ylim(


Corresponding pixels coordinates:  845.9426255990527 839.485470553359
┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 28.82                      │              Nfcn = 727              │
│ EDM = 1.17e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │     Covariance FORCED pos. def.      │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬─────

## Mach numbers

In [9]:
Mach_numbers(PSZ_edges, "XMM")

S edge
  Compression ratio 1.45±0.08
  Mach number 1.31±0.06
  Mach number (g=4/3) 1.25±0.04
  Expected temperature jump (shock) 1.30±0.06
  Expected temperature jump (cold front) 0.69±0.04

NW edge
  Compression ratio 1.52±0.18
  Mach number 1.36±0.13
  Mach number (g=4/3) 1.29±0.10
  Expected temperature jump (shock) 1.35±0.13
  Expected temperature jump (cold front) 0.66±0.08

N edge
  Compression ratio 1.41±0.14
  Mach number 1.28±0.10
  Mach number (g=4/3) 1.23±0.08
  Expected temperature jump (shock) 1.27±0.10
  Expected temperature jump (cold front) 0.71±0.07

SW relic
  Compression ratio 1.00±0.15
  Mach number <1.50
  Mach number (g=4/3) <1.44
  Expected temperature jump (shock) 1.00±0.10
  Expected temperature jump (cold front) 1.00±0.15

NE relic
  Compression ratio 1.00±0.12
  Mach number <1.40
  Mach number (g=4/3) <1.35
  Expected temperature jump (shock) 1.00±0.08
  Expected temperature jump (cold front) 1.00±0.12

